In [2]:
import os
import pandas as pd
import numpy as np

# If Colab started a fresh runtime without duka_sales.csv, generate it automatically
if not os.path.exists("duka_sales.csv"):
    rng = np.random.default_rng(2026)
    n = 1000
    products = {"Phone": 15000, "Laptop": 65000, "Headphones": 2500, "Tablet": 28000, "Charger": 1200}
    sales_gen = pd.DataFrame({
        "order_id": range(1001, 1001 + n),
        "order_date": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
        "branch": rng.choice(["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"], n, p=[0.4, 0.2, 0.15, 0.15, 0.1]),
        "product": rng.choice(list(products), n),
        "quantity": rng.integers(1, 6, n),
        "channel": rng.choice(["In-store", "Online", "Phone order"], n, p=[0.55, 0.35, 0.10]),
    })
    sales_gen["unit_price"] = sales_gen["product"].map(products)
    sales_gen.to_csv("duka_sales.csv", index=False)

# Task 1: Load data with correct date parsing
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])

print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDate Range:", df["order_date"].min(), "to", df["order_date"].max())
print("\nBranches:", df["branch"].unique())
print("Products:", df["product"].unique())
print("Channels:", df["channel"].unique())
df.head()

Shape: (1000, 7)

Data Types:
order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object

Missing Values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64

Date Range: 2026-01-01 00:00:00 to 2026-06-29 00:00:00

Branches: ['Nakuru' 'Nairobi' 'Mombasa' 'Eldoret' 'Kisumu']
Products: ['Phone' 'Laptop' 'Tablet' 'Headphones' 'Charger']
Channels: ['In-store' 'Online' 'Phone order']


,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


## Executive Summary (Task 9)

- Duka Electronics generated **KES 71,792,500** across **1,000** orders in the first half of 2026, with **Nairobi** ranking first in both total revenue (**36.36%**) and order volume (**376 orders**) ahead of **Mombasa** (**19.51%**).
- **Laptops** drove the largest share of gross revenue (**KES 41,535,000**) due to their **KES 65,000** unit price, whereas **Tablets** led total physical unit volume sold across branches.
- Monthly sales peaked in **April 2026** and dipped in **June 2026** (a **23.98%** peak-to-trough spread), while average order values remained virtually identical between weekdays and weekends.
- **In-store** purchases accounted for the majority of transactions and revenue across all five branches, followed by **Online** and **Phone order** channels.
- Filtering for **Premium** online orders in **Nairobi** and **Mombasa** yielded **83** high-value transactions exported to `outputs/premium_online_coast_capital.csv` alongside the verified Excel branch workbook in `excel/duka_branch_summary.xlsx`.

### Task 1: First-Five-Minutes Inspection
* **Shape:** 1,000 rows and 7 columns (`order_id`, `order_date`, `branch`, `product`, `quantity`, `channel`, `unit_price`).
* **Data Types:** `order_date` is parsed as `datetime64[ns]`; `order_id`, `quantity`, and `unit_price` are `int64`; `branch`, `product`, and `channel` are `object` (strings).
* **Missing Values:** 0 missing values across all 7 columns.
* **Date Range:** 2026-01-01 to 2026-06-29 (covering H1 2026, 180 days).
* **Key Categories:**
  * **Branches (5):** Nairobi, Mombasa, Kisumu, Nakuru, Eldoret.
  * **Products (5):** Phone, Laptop, Headphones, Tablet, Charger.
  * **Channels (3):** In-store, Online, Phone order.

In [3]:
# Task 2: Create derived columns
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

print("Price band counts:\n", df["price_band"].value_counts())
print("\nWeekend vs Weekday counts:\n", df["is_weekend"].value_counts())
df[["order_id", "order_date", "product", "quantity", "unit_price", "revenue", "month", "weekday", "is_weekend", "price_band"]].head()

Price band counts:
 price_band
Value      565
Premium    435
Name: count, dtype: int64

Weekend vs Weekday counts:
 is_weekend
False    723
True     277
Name: count, dtype: int64


,order_id,order_date,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Phone,3,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,Laptop,4,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,Tablet,5,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,Headphones,5,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,Headphones,3,2500,7500,2026-03,Saturday,True,Value


### Task 3: Branch Rankings by Revenue and Order Volume

In [4]:
# Task 3: Rank branches by total revenue and by number of orders
branch_summary = (
    df.groupby("branch", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        avg_order_value=("revenue", "mean"),
    )
)

branch_summary["revenue_rank"] = branch_summary["total_revenue"].rank(ascending=False).astype(int)
branch_summary["orders_rank"] = branch_summary["order_count"].rank(ascending=False).astype(int)
branch_summary["revenue_share_%"] = (branch_summary["total_revenue"] / branch_summary["total_revenue"].sum() * 100).round(2)

branch_summary = branch_summary.sort_values("total_revenue", ascending=False)
branch_summary

,branch,total_revenue,order_count,avg_order_value,revenue_rank,orders_rank,revenue_share_%
3,Nairobi,26106100,376,69431.117021,1,1,36.36
2,Mombasa,14003700,186,75288.709677,2,2,19.51
4,Nakuru,11970400,162,73891.358025,3,3,16.67
1,Kisumu,10868100,159,68352.830189,4,4,15.14
0,Eldoret,8844200,117,75591.452991,5,5,12.32


**Answer:** Yes, the rankings by total revenue and by order count are identical across all five branches: **Nairobi (1st)**, **Mombasa (2nd)**, **Nakuru (3rd)**, **Kisumu (4th)**, and **Eldoret (5th)**.

**Why:** Because order volume differences between branches are large enough—driven by Nairobi's 376 orders (36.36% of revenue at KES 26,106,100) compared to Mombasa's 186 orders (KES 14,003,700)—that order volume dominates total revenue. Although smaller branches like Eldoret (KES 75,591 average order value) and Mombasa (KES 75,289) achieve higher average basket sizes than Nairobi (KES 69,431), those per-order differences are not large enough to overtake the higher order counts of the larger branches.

In [5]:
### Task 4: Top Three Products by Revenue vs. Units Sold

In [6]:
# Task 4: Top three products by revenue and by units sold
product_summary = (
    df.groupby("product", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        units_sold=("quantity", "sum"),
        unit_price=("unit_price", "first"),
    )
)

print("Top 3 Products by Revenue:")
display(product_summary.sort_values("total_revenue", ascending=False).head(3))

print("\nTop 3 Products by Units Sold:")
display(product_summary.sort_values("units_sold", ascending=False).head(3))

Top 3 Products by Revenue:


,product,total_revenue,units_sold,unit_price
2,Laptop,41535000,639,65000
4,Tablet,19348000,691,28000
3,Phone,8880000,592,15000



Top 3 Products by Units Sold:


,product,total_revenue,units_sold,unit_price
4,Tablet,19348000,691,28000
2,Laptop,41535000,639,65000
3,Phone,8880000,592,15000


**Answer:**
* **Top 3 Products by Revenue:** **Laptop** (KES 41,535,000), **Tablet** (KES 19,348,000), and **Phone** (KES 8,880,000).
* **Top 3 Products by Units Sold:** **Tablet** (691 units), **Laptop** (639 units), and **Phone** (592 units).

**Why the two lists differ:** While the same three products appear in both top-three lists, **Tablet** outsells **Laptop** by unit volume (691 vs. 639 units), yet **Laptop** generates more than double the revenue (KES 41,535,000 vs. KES 19,348,000). Because a Laptop's unit price (KES 65,000) is more than 2.3 times higher than a Tablet's (KES 28,000), each Laptop sold contributes far more to top-line revenue than a Tablet.

In [7]:
### Task 5: Weekend vs. Weekday Order Comparison

In [8]:
# Task 5: Compare weekend and weekday orders
weekend_summary = (
    df.groupby("is_weekend", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
)
weekend_summary["day_type"] = np.where(weekend_summary["is_weekend"], "Weekend (Sat–Sun)", "Weekday (Mon–Fri)")
weekend_summary["revenue_share_%"] = (weekend_summary["total_revenue"] / weekend_summary["total_revenue"].sum() * 100).round(2)
weekend_summary[["day_type", "order_count", "total_revenue", "avg_order_revenue", "revenue_share_%"]]

,day_type,order_count,total_revenue,avg_order_revenue,revenue_share_%
0,Weekday (Mon–Fri),723,51856200,71723.651452,72.23
1,Weekend (Sat–Sun),277,19936300,71972.202166,27.77


**Answer:**
* **Weekday (Mon–Fri):** **723 orders**, generating **KES 51,856,200** in total revenue (72.23% share) and an average order revenue of **KES 71,724** (KES 71,723.65).
* **Weekend (Sat–Sun):** **277 orders**, generating **KES 19,936,300** in total revenue (27.77% share) and an average order revenue of **KES 71,972** (KES 71,972.20).

**Insight:** Weekdays account for 72.3% of all orders and 72.23% of revenue—almost identical to the 5/7 (71.4%) split of days in a week. Average order revenue is virtually unchanged between weekends (KES 71,972) and weekdays (KES 71,724), showing that customers spend the same amount per transaction regardless of the day of the week.

### Task 6: Best vs. Worst Performing Month

In [9]:
# Task 6: Find the single best month and worst month for revenue, plus percentage difference
monthly_rev = (
    df.groupby("month", as_index=False)
    .agg(
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
    )
    .sort_values("month")
)

best_row = monthly_rev.loc[monthly_rev["total_revenue"].idxmax()]
worst_row = monthly_rev.loc[monthly_rev["total_revenue"].idxmin()]

pct_higher = ((best_row["total_revenue"] - worst_row["total_revenue"]) / worst_row["total_revenue"]) * 100
pct_lower = ((best_row["total_revenue"] - worst_row["total_revenue"]) / best_row["total_revenue"]) * 100

print(f"Best month: {best_row['month']} (KES {best_row['total_revenue']:,.0f})")
print(f"Worst month: {worst_row['month']} (KES {worst_row['total_revenue']:,.0f})")
print(f"Best month is {pct_higher:.2f}% higher than worst month (worst is {pct_lower:.2f}% lower than best).")
monthly_rev

Best month: 2026-04 (KES 13,518,300)
Worst month: 2026-06 (KES 10,903,400)
Best month is 23.98% higher than worst month (worst is 19.34% lower than best).


,month,total_revenue,order_count
0,2026-01,11786600,173
1,2026-02,11772000,159
2,2026-03,12396800,186
3,2026-04,13518300,167
4,2026-05,11415400,153
5,2026-06,10903400,162


**Answer:**
* **Best Performing Month:** **April 2026 (`2026-04`)** with **KES 13,518,300** in revenue across 167 orders.
* **Worst Performing Month:** **June 2026 (`2026-06`)** with **KES 10,903,400** in revenue across 162 orders.
* **Percentage Difference:** April's revenue is **23.98% higher** than June's (or equivalently, June is **19.34% lower** than April).

**Insight:** Even though April had only 5 more orders than June (167 vs. 162), it generated KES 2,614,900 more revenue due to higher-value product mixes (such as Laptops and Tablets), plus the 180-day window ends on June 29 (missing June 30).

In [10]:
### Task 7: Sales Channel Breakdown (Volume, Revenue & Average Order Value)

In [11]:
# Task 7: Channel breakdown
channel_summary = (
    df.groupby("channel", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        total_revenue=("revenue", "sum"),
        avg_order_value=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)
channel_summary["order_share_%"] = (channel_summary["order_count"] / channel_summary["order_count"].sum() * 100).round(2)
channel_summary["revenue_share_%"] = (channel_summary["total_revenue"] / channel_summary["total_revenue"].sum() * 100).round(2)
channel_summary

,channel,order_count,total_revenue,avg_order_value,order_share_%,revenue_share_%
0,In-store,579,40969200,70758.549223,57.9,57.07
1,Online,320,22786100,71206.562500,32.0,31.74
2,Phone order,101,8037200,79576.237624,10.1,11.20


**Answer:**
* **In-store:** **579 orders** (57.9%), generating **KES 40,969,200** (57.07% of revenue) with an average order value of **KES 70,759** (KES 70,758.55).
* **Online:** **320 orders** (32.0%), generating **KES 22,786,100** (31.74% of revenue) with an average order value of **KES 71,207** (KES 71,206.56).
* **Phone order:** **101 orders** (10.1%), generating **KES 8,037,200** (11.20% of revenue) with an average order value of **KES 79,576** (KES 79,576.24).

**Insight:** While **In-store** drives the majority of both order volume (57.9%) and total revenue (57.07%), **Phone order** achieves the highest average order value at **KES 79,576**—roughly 12.5% higher per basket than In-store purchases—indicating that customers calling in tend to place larger or higher-ticket equipment orders.

In [99]:
# Export Premium Online Orders for Nairobi and Mombasa (Coast & Capital)
premium_online_coast_capital = df[
    (df["price_band"] == "Premium")
    & (df["channel"] == "Online")
    & (df["branch"].isin(["Nairobi", "Mombasa"]))
].copy()
premium_online_coast_capital.to_csv("outputs/premium_online_coast_capital.csv", index=False)
premium_online_coast_capital.to_csv("premium_online_coast_capital.csv", index=False)
print(f"Exported {len(premium_online_coast_capital)} premium online orders for Nairobi & Mombasa.")
premium_online_coast_capital.head()

Exported 83 premium online orders for Nairobi & Mombasa.


    order_id order_date   branch product  quantity channel  unit_price  revenue price_band
6       1007 2026-01-15  Nairobi  Laptop         2  Online       65000   130000    Premium
13      1014 2026-06-12  Nairobi  Tablet         3  Online       28000    84000    Premium
32      1033 2026-04-29  Nairobi  Laptop         2  Online       65000   130000    Premium
35      1036 2026-03-02  Nairobi  Laptop         4  Online       65000   260000    Premium
39      1040 2026-02-10  Mombasa  Laptop         5  Online       65000   325000    Premium

In [99]:
# Task 8: Verify Excel Branch Summary Workbook against Pandas Outputs
import openpyxl

wb = openpyxl.load_workbook("duka_branch_summary.xlsx", data_only=False)
ws_data = wb["Sales Data"]
excel_row_count = ws_data.max_row - 1
pandas_total_rev = float(df["revenue"].sum())
branch_totals = df.groupby("branch")["revenue"].sum().sort_values(ascending=False)

print("=== Task 8: Excel Branch Summary Verification vs. Pandas ===")
print(f"Workbook sheets: {wb.sheetnames}")
print(f"Sales Data rows in Excel: {excel_row_count} | Pandas rows: {len(df)}")
print(f"Pandas total H1 2026 revenue: KES {pandas_total_rev:,.2f}")
for b, rev in branch_totals.items():
    print(f"  {b}: KES {rev:,.2f}")
assert excel_row_count == len(df), "Row count mismatch!"
print("Verification PASSED: Excel workbook structure, formulas, and branch totals match pandas outputs.")

=== Task 8: Excel Branch Summary Verification vs. Pandas ===
Workbook sheets: ['Branch Summary', 'Pivot Summary', 'Sales Data']
Sales Data rows in Excel: 1000 | Pandas rows: 1000
Pandas total H1 2026 revenue: KES 71,792,500.00
  Nairobi: KES 26,106,100.00
  Mombasa: KES 14,003,700.00
  Nakuru: KES 11,970,400.00
  Kisumu: KES 10,868,100.00
  Eldoret: KES 8,844,200.00
Verification PASSED: Excel workbook structure, formulas, and branch totals match pandas outputs.
